In [8]:
import numpy as np
import matplotlib.pyplot as plt

def show(image, title: str = None, cmap: str = None):
    plt.imshow(image, cmap=cmap)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
from pathlib import Path

ds_path = Path('S1_v2')


In [ ]:
from petroscope.segmentation.classes import ClassSet, LumenStoneClasses

classset = LumenStoneClasses.S1()
for cl in classset.classes:
    print(cl)

Сгенерируем патчи размера 512x512 с наложением в 128 пикселей

In [ ]:
PATCH_SIZE = 512
PATCH_STRIDE = 384
SPLIT_RATIO_TRAIN = 0.8 # 80% для тренировки, 20% для валидации

dsp_path = Path(f"S1_v2_{PATCH_SIZE}_{PATCH_STRIDE}_{int(SPLIT_RATIO_TRAIN * 100)}") # путь к папке с патчами

In [ ]:
def save_patch(im, mask, i, x0, y0, patches_path, masks_path, patches_valid_path=None, masks_valid_path=None):
    if np.random.rand() < 1 - SPLIT_RATIO_TRAIN:
        patch = im.crop((x0, y0, x0 + PATCH_SIZE, y0 + PATCH_SIZE))
        patch.save(patches_valid_path/f"{i}_{x0}_{y0}.jpg")

        patch_mask = mask.crop((x0, y0, x0 + PATCH_SIZE, y0 + PATCH_SIZE))
        patch_mask.save(masks_valid_path/f"{i}_{x0}_{y0}.png")
    else:
        patch = im.crop((x0, y0, x0 + PATCH_SIZE, y0 + PATCH_SIZE))
        patch.save(patches_path/f"{i}_{x0}_{y0}.jpg")

        patch_mask = mask.crop((x0, y0, x0 + PATCH_SIZE, y0 + PATCH_SIZE))
        patch_mask.save(masks_path/f"{i}_{x0}_{y0}.png")

def save_patch_test(im, mask, i, x0, y0, patches_path, masks_path):
    patch = im.crop((x0, y0, x0 + PATCH_SIZE, y0 + PATCH_SIZE))
    patch.save(patches_path/f"{i}_{x0}_{y0}.jpg")

    patch_mask = mask.crop((x0, y0, x0 + PATCH_SIZE, y0 + PATCH_SIZE))
    patch_mask.save(masks_path/f"{i}_{x0}_{y0}.png")

In [ ]:
patches_train_path = dsp_path/"patches"/"train"
patches_valid_path = dsp_path/"patches"/"valid"
masks_train_path = dsp_path/"masks"/"train"
masks_valid_path = dsp_path/"masks"/"valid"

In [7]:
from PIL import Image


patches_train_path.mkdir(parents=True, exist_ok=True)
patches_valid_path.mkdir(parents=True, exist_ok=True)
masks_train_path.mkdir(parents=True, exist_ok=True)
masks_valid_path.mkdir(parents=True, exist_ok=True)


for i in range(1, 65):
    im = Image.open(ds_path/"imgs"/"train"/f"train_{0 if i < 10 else ''}{i}.jpg")
    mask = Image.open(ds_path/"masks"/"train"/f"train_{0 if i < 10 else ''}{i}.png")
    xs, ys = im.size
    for x0 in range(0, xs - PATCH_SIZE + 1, PATCH_STRIDE):
        for y0 in range(0, ys - PATCH_SIZE + 1, PATCH_STRIDE):
            save_patch(im, mask, i, x0, y0, patches_train_path, masks_train_path, patches_valid_path, masks_valid_path)

        if ys % PATCH_STRIDE != 0:
            save_patch(im, mask, i, x0, ys - PATCH_SIZE, patches_train_path, masks_train_path, patches_valid_path, masks_valid_path)

    if xs % PATCH_STRIDE != 0:
        for y0 in range(0, ys - PATCH_SIZE + 1, PATCH_STRIDE):
            save_patch(im, mask, i, xs - PATCH_SIZE, y0, patches_train_path, masks_train_path, patches_valid_path, masks_valid_path)

        if ys % PATCH_STRIDE != 0:
            save_patch(im, mask, i, xs - PATCH_SIZE, ys - PATCH_SIZE, patches_train_path, masks_train_path, patches_valid_path, masks_valid_path)

patches_test_path = dsp_path/"patches"/"test"
masks_test_path = dsp_path/"masks"/"test"

patches_test_path.mkdir(parents=True, exist_ok=True)
masks_test_path.mkdir(parents=True, exist_ok=True)


for i in range(1, 21):
    im = Image.open(ds_path/"imgs"/"test"/f"test_{0 if i < 10 else ''}{i}.jpg")
    mask = Image.open(ds_path/"masks"/"test"/f"test_{0 if i < 10 else ''}{i}.png")
    xs, ys = im.size
    for x0 in range(0, xs - PATCH_SIZE + 1, PATCH_STRIDE):
        for y0 in range(0, ys - PATCH_SIZE + 1, PATCH_STRIDE):
            save_patch_test(im, mask, i, x0, y0, patches_test_path, masks_test_path)
        if ys % PATCH_STRIDE != 0:
            save_patch_test(im, mask, i, x0, ys - PATCH_SIZE, patches_test_path, masks_test_path)
    if xs % PATCH_STRIDE != 0:
        for y0 in range(0, ys - PATCH_SIZE + 1, PATCH_STRIDE):
            save_patch_test(im, mask, i, xs - PATCH_SIZE, y0, patches_test_path, masks_test_path)
        if ys % PATCH_STRIDE != 0:
            save_patch_test(im, mask, i, xs - PATCH_SIZE, ys - PATCH_SIZE, patches_test_path, masks_test_path)
    

Сгенерируем датасет из однородных патчей

In [9]:
import cv2


homo_dsp_path = Path(f"S1_v2_homogeneous")

patches_train_homogeneous_path = homo_dsp_path/"patches"/"train"
patches_valid_homogeneous_path = homo_dsp_path/"patches"/"valid"
masks_train_homogeneous_path = homo_dsp_path/"masks"/"train"
masks_valid_homogeneous_path = homo_dsp_path/"masks"/"valid"

patches_train_homogeneous_path.mkdir(parents=True, exist_ok=True)
patches_valid_homogeneous_path.mkdir(parents=True, exist_ok=True)
masks_train_homogeneous_path.mkdir(parents=True, exist_ok=True)
masks_valid_homogeneous_path.mkdir(parents=True, exist_ok=True)

homogeneous_cnt = 0

for img_p in patches_train_path.iterdir():
    im = cv2.imread(str(img_p))
    if im.std() < 10:
        homogeneous_cnt += 1
        cv2.imwrite(str(patches_train_homogeneous_path/img_p.name), im)
        mask_p = masks_train_path/f"{img_p.stem}.png"
        cv2.imwrite(str(masks_train_homogeneous_path/f"{img_p.stem}.png"), cv2.imread(str(mask_p)))

print("Amount of homogeneous patches in training set:", homogeneous_cnt)
homogeneous_cnt = 0

for img_p in patches_valid_path.iterdir():
    im = cv2.imread(str(img_p))
    if im.std() < 10:
        homogeneous_cnt += 1
        cv2.imwrite(str(patches_valid_homogeneous_path/img_p.name), im)
        mask_p = masks_valid_path/f"{img_p.stem}.png"
        cv2.imwrite(str(masks_valid_homogeneous_path/f"{img_p.stem}.png"), cv2.imread(str(mask_p)))

print("Amount of homogeneous patches in validation set:", homogeneous_cnt)

Amount of homogeneous patches in training set: 243
Amount of homogeneous patches in validation set: 62


Аугментируем датасет из однородных патчей:
повороты
отражения

In [10]:
for img_p in patches_train_homogeneous_path.iterdir():
    im = cv2.imread(str(img_p))
    im1 = cv2.rotate(im, cv2.ROTATE_90_CLOCKWISE)
    for i in range(3):
        cv2.imwrite(str(patches_train_homogeneous_path/f"{img_p.stem}_rot{i}.jpg"), im1)
        im1 = cv2.rotate(im1, cv2.ROTATE_90_CLOCKWISE)
    cv2.imwrite(str(patches_train_homogeneous_path/f"{img_p.stem}_flip0.jpg"), cv2.flip(im, -1))
    cv2.imwrite(str(patches_train_homogeneous_path/f"{img_p.stem}_flip1.jpg"), cv2.flip(im, 0))
    cv2.imwrite(str(patches_train_homogeneous_path/f"{img_p.stem}_flip2.jpg"), cv2.flip(im, 1))

    mask = cv2.imread(str(masks_train_homogeneous_path/f"{img_p.stem}.png"))
    mask1 = cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    for i in range(3):
        cv2.imwrite(str(masks_train_homogeneous_path/f"{img_p.stem}_rot{i}.png"), mask1)
        mask1 = cv2.rotate(mask1, cv2.ROTATE_90_CLOCKWISE)

    cv2.imwrite(str(masks_train_homogeneous_path/f"{img_p.stem}_flip0.png"), cv2.flip(mask, -1))
    cv2.imwrite(str(masks_train_homogeneous_path/f"{img_p.stem}_flip1.png"), cv2.flip(mask, 0))
    cv2.imwrite(str(masks_train_homogeneous_path/f"{img_p.stem}_flip2.png"), cv2.flip(mask, 1))

for img_p in patches_valid_homogeneous_path.iterdir():
    im = cv2.imread(str(img_p))
    im1 = cv2.rotate(im, cv2.ROTATE_90_CLOCKWISE)
    for i in range(3):
        cv2.imwrite(str(patches_valid_homogeneous_path/f"{img_p.stem}_rot{i}.jpg"), im1)
        im1 = cv2.rotate(im1, cv2.ROTATE_90_CLOCKWISE)
    cv2.imwrite(str(patches_valid_homogeneous_path/f"{img_p.stem}_flip0.jpg"), cv2.flip(im, -1))
    cv2.imwrite(str(patches_valid_homogeneous_path/f"{img_p.stem}_flip1.jpg"), cv2.flip(im, 0))
    cv2.imwrite(str(patches_valid_homogeneous_path/f"{img_p.stem}_flip2.jpg"), cv2.flip(im, 1))

    mask = cv2.imread(str(masks_valid_homogeneous_path/f"{img_p.stem}.png"))
    mask1 = cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    for i in range(3):
        cv2.imwrite(str(masks_valid_homogeneous_path/f"{img_p.stem}_rot{i}.png"), mask1)
        mask1 = cv2.rotate(mask1, cv2.ROTATE_90_CLOCKWISE)

    cv2.imwrite(str(masks_valid_homogeneous_path/f"{img_p.stem}_flip0.png"), cv2.flip(mask, -1))
    cv2.imwrite(str(masks_valid_homogeneous_path/f"{img_p.stem}_flip1.png"), cv2.flip(mask, 0))
    cv2.imwrite(str(masks_valid_homogeneous_path/f"{img_p.stem}_flip2.png"), cv2.flip(mask, 1))




In [ ]:
import torch
import torch.nn as nn

def double_convolution(in_channels, out_channels):
    conv_op = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True)
    )
    return conv_op

In [ ]:
def log_memory(step):
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"Step {step}: Allocated={allocated:.2f}GB, Reserved={reserved:.2f}GB")



In [ ]:
from petroscope.segmentation.eval import SegmDetailedTester

def test(model, test_img_mask, test_log_dir):
    tester = SegmDetailedTester(
        out_dir=Path(test_log_dir),
        classes=classset,
        max_classes=LumenStoneClasses.max_classes(),
        void_pad=0,
        void_border_width=4,
        vis_segmentation=True,
    )
    res, res_void = tester.test_on_set(
        test_img_mask,
        predict_func=model.predict_image,
        epoch=0,
    )
    print("results without void borders:\n", res)
    print("results with void borders:\n", res_void)

In [ ]:
import petroscope.segmentation as segm
from pathlib import Path
from typing import Iterable
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm
from petroscope.segmentation.classes import ClassSet, LumenStoneClasses
from petroscope.segmentation.utils import load_image, load_mask

class SegmentationDataset(Dataset):
    def __init__(
        self,
        img_mask_paths: Iterable[tuple[Path, Path]],
        code_to_idx: dict[int, int],
        image_size: int,
    ) -> None:
        self.img_mask_paths = list(img_mask_paths)
        self.code_to_idx = code_to_idx
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.img_mask_paths)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        img_p, mask_p = self.img_mask_paths[index]
        image = load_image(img_p, normalize=True).astype(np.float32)
        mask = load_mask(mask_p)

        encoded_mask = np.zeros(mask.shape, dtype=np.int64)
        for code, class_idx in self.code_to_idx.items():
            encoded_mask[mask == code] = class_idx

        image_tensor = torch.from_numpy(np.transpose(image, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(encoded_mask).long()
        return image_tensor, mask_tensor


class UnetModel(segm.GeoSegmModel, nn.Module):
    def __init__(self, classes: ClassSet) -> None:
        super().__init__()
        nn.Module.__init__(self)
        self.classes = classes
        self.class_codes = [cl.code for cl in classes.classes]
        self.code_to_idx = {code: idx for idx, code in enumerate(self.class_codes)}
        self.idx_to_code = {idx: code for code, idx in self.code_to_idx.items()}
        self.code_to_color = {cl.code: cl.color for cl in classes.classes}
        self.num_classes = len(self.class_codes)
        self.max_classes = LumenStoneClasses.max_classes()
        self.image_size = PATCH_SIZE
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Device: ", self.device)

        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down_convolution_1 = double_convolution(3, 64)
        self.down_convolution_2 = double_convolution(64, 128)
        self.down_convolution_3 = double_convolution(128, 256)
        self.down_convolution_4 = double_convolution(256, 512)
        self.down_convolution_5 = double_convolution(512, 1024)

        self.up_transpose_1 = nn.ConvTranspose2d(
            in_channels=1024, out_channels=512,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_1 = double_convolution(1024, 512)
        self.up_transpose_2 = nn.ConvTranspose2d(
            in_channels=512, out_channels=256,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_2 = double_convolution(512, 256)
        self.up_transpose_3 = nn.ConvTranspose2d(
            in_channels=256, out_channels=128,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_3 = double_convolution(256, 128)
        self.up_transpose_4 = nn.ConvTranspose2d(
            in_channels=128, out_channels=64,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_4 = double_convolution(128, 64)
        self.out = nn.Conv2d(
            in_channels=64, out_channels=self.num_classes,
            kernel_size=1,
        )

        self.to(self.device)

    def load(self, saved_path: Path, **kwargs) -> None:
        checkpoint = torch.load(saved_path, map_location=self.device)
        self.load_state_dict(checkpoint["state_dict"])
        self.image_size = int(checkpoint.get("image_size", self.image_size))
        loaded_code_to_idx = checkpoint.get("code_to_idx")
        if loaded_code_to_idx is not None:
            self.code_to_idx = {int(code): int(idx) for code, idx in loaded_code_to_idx.items()}
            self.idx_to_code = {idx: code for code, idx in self.code_to_idx.items()}
        loaded_code_to_color = checkpoint.get("code_to_color")
        if loaded_code_to_color is not None:
            self.code_to_color = loaded_code_to_color
        self.to(self.device)
        nn.Module.train(self, False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        down_1 = self.down_convolution_1(x)
        down_2 = self.max_pool2d(down_1)
        down_3 = self.down_convolution_2(down_2)
        down_4 = self.max_pool2d(down_3)
        down_5 = self.down_convolution_3(down_4)
        down_6 = self.max_pool2d(down_5)
        down_7 = self.down_convolution_4(down_6)
        down_8 = self.max_pool2d(down_7)
        down_9 = self.down_convolution_5(down_8)

        up_1 = self.up_transpose_1(down_9)
        x = self.up_convolution_1(torch.cat([down_7, up_1], dim=1))

        up_2 = self.up_transpose_2(x)
        x = self.up_convolution_2(torch.cat([down_5, up_2], dim=1))

        up_3 = self.up_transpose_3(x)
        x = self.up_convolution_3(torch.cat([down_3, up_3], dim=1))

        up_4 = self.up_transpose_4(x)
        x = self.up_convolution_4(torch.cat([down_1, up_4], dim=1))
        return self.out(x)

    def validate(
        self,
        img_valid_paths: Iterable[tuple[Path, Path]],
        batch_size: int = 2,
        num_workers: int = 0,
    ) -> float:
        img_valid_paths = list(img_valid_paths)
        if not img_valid_paths:
            print("Validation skipped: no validation patches provided")
            return float("nan")

        dataset = SegmentationDataset(
            img_mask_paths=img_valid_paths,
            code_to_idx=self.code_to_idx,
            image_size=self.image_size,
        )
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
        )

        criterion = nn.CrossEntropyLoss()
        total_loss = 0.0
        total_count = 0

        self.to(self.device)
        nn.Module.train(self, False)
        with torch.no_grad():
            progress = tqdm(loader, desc="Validation", leave=False)
            for images, masks in progress:
                images = images.to(self.device)
                masks = masks.to(self.device)
                logits = self(images)
                loss = criterion(logits, masks)

                bs = images.size(0)
                total_loss += loss.item() * bs
                total_count += bs
                progress.set_postfix(loss=f"{loss.item():.4f}")

        if total_count == 0:
            return float("nan")

        val_loss = total_loss / total_count
        print(f"Validation loss: {val_loss:.4f}")
        return val_loss

    def train(
        self, img_mask_paths: Iterable[tuple[Path, Path]], img_valid_paths: Iterable[tuple[Path, Path]], **kwargs
    ) -> None:
        epochs = int(kwargs.get("epochs", 3))
        batch_size = int(kwargs.get("batch_size", 2))
        learning_rate = float(kwargs.get("lr", 1e-3))
        image_size = int(kwargs.get("image_size", PATCH_SIZE))
        num_workers = int(kwargs.get("num_workers", 0))
        save_path = Path(kwargs.get("save_path", "output/unet_model.pt"))
        scheduler_patience = int(kwargs.get("scheduler_patience", 5))
        scheduler_factor = float(kwargs.get("scheduler_factor", 0.5))
        scheduler_min_lr = float(kwargs.get("scheduler_min_lr", 1e-6))
        scheduler_threshold = float(kwargs.get("scheduler_threshold", 1e-4))

        img_mask_paths = list(img_mask_paths)
        if not img_mask_paths:
            raise ValueError("img_mask_paths must contain at least one image-mask pair")

        self.image_size = image_size
        dataset = SegmentationDataset(
            img_mask_paths=img_mask_paths,
            code_to_idx=self.code_to_idx,
            image_size=self.image_size,
        )
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
        )

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=scheduler_factor,
            patience=scheduler_patience,
            threshold=scheduler_threshold,
            min_lr=scheduler_min_lr,
        )

        save_path.parent.mkdir(parents=True, exist_ok=True)

        self.to(self.device)
        for epoch in range(epochs):
            nn.Module.train(self, True)
            epoch_loss = 0.0
            progress = tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}")

            idx = 0
            for images, masks in progress:
                images = images.to(self.device)
                masks = masks.to(self.device)
                optimizer.zero_grad(set_to_none=True)
                logits = self(images)
                loss = criterion(logits, masks)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item() * images.size(0)
                progress.set_postfix(loss=f"{loss.item():.4f}")

                if idx % 40 == 0:
                    log_memory(idx)

                idx += 1

            mean_loss = epoch_loss / len(dataset)
            print(f"Epoch {epoch + 1}/{epochs}: train_loss={mean_loss:.4f}")

            val_loss = self.validate(
                img_valid_paths=img_valid_paths,
                batch_size=batch_size,
                num_workers=num_workers,
            )

            monitor_loss = val_loss if not np.isnan(val_loss) else mean_loss
            prev_lr = optimizer.param_groups[0]["lr"]
            scheduler.step(monitor_loss)
            current_lr = optimizer.param_groups[0]["lr"]
            if current_lr < prev_lr:
                print(f"LR reduced: {prev_lr:.6e} -> {current_lr:.6e}")
            else:
                print(f"LR: {current_lr:.6e}")

            torch.cuda.empty_cache()

            torch.save(
                {
                    "state_dict": self.state_dict(),
                    "image_size": self.image_size,
                    "code_to_idx": self.code_to_idx,
                    "code_to_color": self.code_to_color,
                },
                save_path,
            )
            print(f"Saved model to {save_path}")

        nn.Module.train(self, False)

    def predict_image(self, image: np.ndarray) -> np.ndarray:
        original_height, original_width = image.shape[:2]
        if image.dtype == np.uint8:
            normalized_image = image.astype(np.float32) / 255.0
        else:
            normalized_image = image.astype(np.float32)
            if normalized_image.max() > 1.0:
                normalized_image /= 255.0

        # Accumulate softmax probabilities for overlapping patches
        prob_sum = np.zeros((original_height, original_width, self.num_classes), dtype=np.float32)
        count = np.zeros((original_height, original_width), dtype=np.float32)

        softmax = nn.Softmax(dim=1)
        self.to(self.device)
        nn.Module.train(self, False)

        # Build list of patch start positions, covering edges
        def get_starts(total, patch, stride):
            starts = list(range(0, total - patch + 1, stride))
            if not starts or starts[-1] + patch < total:
                starts.append(max(0, total - patch))
            return starts

        x_starts = get_starts(original_width, self.image_size, PATCH_STRIDE)
        y_starts = get_starts(original_height, self.image_size, PATCH_STRIDE)

        for x0 in x_starts:
            for y0 in y_starts:
                patch = normalized_image[y0:y0 + self.image_size, x0:x0 + self.image_size]
                patch_tensor = torch.from_numpy(
                    np.transpose(patch, (2, 0, 1))
                ).unsqueeze(0).float().to(self.device)

                with torch.no_grad():
                    logits = self(patch_tensor)
                    probs = softmax(logits).squeeze(0).cpu().numpy()  # (num_classes, H, W)

                probs = np.transpose(probs, (1, 2, 0))  # (H, W, num_classes)
                prob_sum[y0:y0 + self.image_size, x0:x0 + self.image_size] += probs
                count[y0:y0 + self.image_size, x0:x0 + self.image_size] += 1.0

        # Average over overlapping patches and take argmax
        prob_avg = prob_sum / np.maximum(count[:, :, np.newaxis], 1.0)
        pred_indices = np.argmax(prob_avg, axis=2)  # (H, W)

        pred_codes = np.zeros(pred_indices.shape, dtype=np.uint8)
        for class_idx, code in self.idx_to_code.items():
            pred_codes[pred_indices == class_idx] = code

        prediction = np.zeros(
            (original_height, original_width, self.max_classes),
            dtype=np.float32,
        )
        ys, xs = np.indices((original_height, original_width))
        prediction[ys, xs, pred_codes] = 1.0
        return prediction


In [ ]:
train_img_mask_p = [
    (img_p, dsp_path / "masks" / "train" / f"{img_p.stem}.png")
    for img_p in sorted((dsp_path / "patches" / "train").iterdir())
]
valid_img_mask_p = [
    (img_p, dsp_path / "masks" / "valid" / f"{img_p.stem}.png")
    for img_p in sorted((dsp_path / "patches" / "valid").iterdir())
]
test_img_mask_p = [
    (img_p, dsp_path / "masks" / "test" / f"{img_p.stem}.png")
    for img_p in sorted((dsp_path / "patches" / "test").iterdir())
]



train_img_mask = [
    (img_p, ds_path / "masks" / "train" / f"{img_p.stem}.png")
    for img_p in sorted((ds_path / "imgs" / "train").iterdir())
]
test_img_mask = [
    (img_p, ds_path / "masks" / "test" / f"{img_p.stem}.png")
    for img_p in sorted((ds_path / "imgs" / "test").iterdir())
]

In [ ]:
from torchsummary import summary

model = UnetModel(classes=classset)
summary(model, (3, PATCH_SIZE, PATCH_SIZE))

In [ ]:
model = UnetModel(classes=classset)
model.load("unet_model.pt")

In [ ]:
torch.cuda.empty_cache()
model.train(
    img_mask_paths=train_img_mask_p,
    img_valid_paths=valid_img_mask_p,
    epochs=50,
    batch_size=8,
    image_size=PATCH_SIZE,
    lr=3e-4,
    scheduler_patience=2,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    scheduler_threshold=1e-4,
    save_path=Path("output/unet_model.pt"),
)
print("Trained classes:\n", "\n".join([f"{code}: {color}" for code, color in model.code_to_color.items()]))

In [ ]:
test(model, test_img_mask, "output/test")

tensorboard
mlflow